# 🚀 7. Feature Engineering (RGB + AVG + NTSC) — Fusi Fitur Custom CNN & Training SVM

> 💡 **Penjelasan Arsitektur (Custom CNN):**
> Representasi fitur 1.536-D pada notebook ini diperoleh dari fusi 3 **Custom CNN murni** terlatih (RGB 512-D + AVG 512-D + NTSC 512-D) yang semuanya dilatih dari nol tanpa memanfaatkan backbone pretrained (seperti ResNet/VGG).

Notebook ini mengimplementasikan Feature Engineering melalui **Multi-Domain Feature Fusion**:
1. Menggabungkan fitur tingkat tinggi dari 3 model CNN terlatih:
   - Fitur RGB: 512 dimensi (informasi warna kromatik)
   - Fitur Grayscale AVG: 512 dimensi (informasi intensitas seragam)
   - Fitur Grayscale NTSC: 512 dimensi (informasi luminansi perseptual)
   - **Total Fitur Gabungan: 1.536 Dimensi** ($512 + 512 + 512$).
2. Preprocessing & Feature Weighting multi-domain (per-domain scaling, bobot fitur, dan L2 normalizer).
3. Pelatihan SVM Classifier pada ruang fitur gabungan 1.536 dimensi dan penyimpanan model final.

In [1]:
import os
import sys
import numpy as np


ARTIFACT_DIR = 'cifar10_artifacts/rgb_avg_ntsc'
os.makedirs(ARTIFACT_DIR, exist_ok=True)
print(f"Artifact directory: {ARTIFACT_DIR}")

# Lokasi fitur dari ketiga model CNN
rgb_dir  = 'cifar10_artifacts/rgb'
avg_dir  = 'cifar10_artifacts/grayscale_avg'
ntsc_dir = 'cifar10_artifacts/grayscale_ntsc'

print("Memuat vektor fitur training dan testing dari 3 workflow sebelumnya...")
rgb_train  = np.load(os.path.join(rgb_dir, 'train_features.npz'))
avg_train  = np.load(os.path.join(avg_dir, 'train_features.npz'))
ntsc_train = np.load(os.path.join(ntsc_dir, 'train_features.npz'))

rgb_test   = np.load(os.path.join(rgb_dir, 'test_features.npz'))
avg_test   = np.load(os.path.join(avg_dir, 'test_features.npz'))
ntsc_test  = np.load(os.path.join(ntsc_dir, 'test_features.npz'))

y_train = rgb_train['y_train']
y_test  = rgb_test['y_test']


Artifact directory: cifar10_artifacts/rgb_avg_ntsc
Memuat vektor fitur training dan testing dari 3 workflow sebelumnya...


## 📥 1. Feature Fusion (Concatenation) Menjadi Vektor 1.536-Dimensi


In [2]:
from sklearn.model_selection import train_test_split

RANDOM_STATE = 23092026

def get_cifar10_partitions(y_train_labels):
    indices_all = np.arange(len(y_train_labels))
    idx_cnn_pool, idx_svm_val = train_test_split(
        indices_all,
        test_size=5000,
        random_state=RANDOM_STATE,
        stratify=y_train_labels
    )
    idx_cnn_train, idx_cnn_val = train_test_split(
        idx_cnn_pool,
        test_size=5000,
        random_state=RANDOM_STATE,
        stratify=y_train_labels[idx_cnn_pool]
    )
    return idx_cnn_train, idx_cnn_val, idx_svm_val

print("Menggabungkan fitur training (RGB + AVG + NTSC)...")
X_train_fused = np.concatenate([
    rgb_train['X_train_features'],
    avg_train['X_train_features'],
    ntsc_train['X_train_features']
], axis=1)

print("Menggabungkan fitur testing (RGB + AVG + NTSC)...")
X_test_fused = np.concatenate([
    rgb_test['X_test_features'],
    avg_test['X_test_features'],
    ntsc_test['X_test_features']
], axis=1)

if 'idx_cnn_train' in rgb_train and 'idx_svm_val' in rgb_train:
    idx_cnn_train = rgb_train['idx_cnn_train']
    idx_cnn_val = rgb_train['idx_cnn_val']
    idx_svm_val = rgb_train['idx_svm_val']
else:
    idx_cnn_train, idx_cnn_val, idx_svm_val = get_cifar10_partitions(y_train)

print()
print(f"[SELESAI] X_train_fused shape : {X_train_fused.shape} (50.000 sampel, 1.536 dimensi)")
print(f"[SELESAI] X_test_fused shape  : {X_test_fused.shape} (10.000 sampel, 1.536 dimensi)")
print(f"  - CNN Training Partition   : {len(idx_cnn_train)} sampel (40k)")
print(f"  - CNN Validation Partition : {len(idx_cnn_val)} sampel (5k)")
print(f"  - Independent SVM Val Part.: {len(idx_svm_val)} sampel (5k unseen oleh CNN)")


Menggabungkan fitur training (RGB + AVG + NTSC)...
Menggabungkan fitur testing (RGB + AVG + NTSC)...

[SELESAI] X_train_fused shape : (50000, 1536) (50.000 sampel, 1.536 dimensi)
[SELESAI] X_test_fused shape  : (10000, 1536) (10.000 sampel, 1.536 dimensi)
  - CNN Training Partition   : 40000 sampel (40k)
  - CNN Validation Partition : 5000 sampel (5k)
  - Independent SVM Val Part.: 5000 sampel (5k unseen oleh CNN)


## 💾 2. Menyimpan Vektor Fitur Gabungan ke Disk (.npz)


In [3]:
train_feat_path = os.path.join(ARTIFACT_DIR, 'train_features.npz')
test_feat_path  = os.path.join(ARTIFACT_DIR, 'test_features.npz')

# Simpan murni fitur gabungan beserta partisi indeks akademik (tanpa cache prediksi)
np.savez_compressed(
    train_feat_path,
    X_train_features=X_train_fused,
    y_train=y_train,
    idx_cnn_train=idx_cnn_train,
    idx_cnn_val=idx_cnn_val,
    idx_svm_val=idx_svm_val
)
np.savez_compressed(test_feat_path, X_test_features=X_test_fused, y_test=y_test)

print(f"[BERHASIL] Fitur gabungan training tersimpan murni: {train_feat_path}")
print(f"[BERHASIL] Fitur gabungan testing tersimpan murni : {test_feat_path}")


[BERHASIL] Fitur gabungan training tersimpan murni: cifar10_artifacts/rgb_avg_ntsc\train_features.npz
[BERHASIL] Fitur gabungan testing tersimpan murni : cifar10_artifacts/rgb_avg_ntsc\test_features.npz


---
## ⚔️ 3. Pelatihan SVM Classifier pada Fitur Gabungan 1.536 Dimensi & Serialisasi Final
Menerapkan FusedDomainPreprocessor dan melatih model SVM Classifier pada seluruh sampel training.

In [4]:
import os
import sys
import pickle
import json
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin, ClassifierMixin
from sklearn.preprocessing import StandardScaler, Normalizer
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

from joblib import Parallel, delayed
ARTIFACT_DIR = 'cifar10_artifacts/rgb_avg_ntsc'
train_feat_path = os.path.join(ARTIFACT_DIR, 'train_features.npz')
print(f"Memuat vektor fitur gabungan training dari: {train_feat_path}")
train_data = np.load(train_feat_path)
X_train_features = train_data['X_train_features']
y_train = train_data['y_train']
print(f"Dimensi fitur gabungan: {X_train_features.shape}")

RANDOM_STATE = 23092026


Memuat vektor fitur gabungan training dari: cifar10_artifacts/rgb_avg_ntsc\train_features.npz
Dimensi fitur gabungan: (50000, 1536)


## 🛠️ 1. Definisi Model, Validation-Driven Search & Pelatihan Model Final (Zero Leakage)


In [5]:
# ==============================================================================
# 1. Custom Transformer: Multi-Domain Feature Preprocessor (1.536-Dimensi)
# ==============================================================================
class FusedDomainPreprocessor(BaseEstimator, TransformerMixin):
    """
    Preprocessor modular untuk fitur gabungan 1.536 dimensi (RGB: 512, AVG: 512, NTSC: 512).
    Mendukung strategi:
      Mode A: 'standard'            -> Global StandardScaler
      Mode B: 'standard_l2'         -> Global StandardScaler + L2 Normalizer
      Mode C: 'per_domain_standard' -> Per-domain StandardScaler
      Mode D: 'per_domain_weighted' -> Per-domain StandardScaler + Feature Weighting (w_rgb, w_avg, w_ntsc)
      Mode E: 'global_weighted'     -> Global StandardScaler + Feature Weighting (w_rgb, w_avg, w_ntsc)
    """
    def __init__(self, mode='per_domain_weighted', weights=(1.0, 0.7, 0.7), apply_l2=True):
        self.mode = mode
        self.weights = weights
        self.apply_l2 = apply_l2
        self.scaler_rgb = None
        self.scaler_avg = None
        self.scaler_ntsc = None
        self.scaler_global = None
        self.normalizer = Normalizer(norm='l2') if apply_l2 else None

    def fit(self, X, y=None):
        if self.mode in ['per_domain_standard', 'per_domain_weighted', 'domain_weighted']:
            self.scaler_rgb = StandardScaler().fit(X[:, :512])
            self.scaler_avg = StandardScaler().fit(X[:, 512:1024])
            self.scaler_ntsc = StandardScaler().fit(X[:, 1024:1536])
        else:
            self.scaler_global = StandardScaler().fit(X)
        return self

    def transform(self, X):
        w_rgb, w_avg, w_ntsc = self.weights
        if self.mode in ['per_domain_standard', 'per_domain_weighted', 'domain_weighted']:
            x_rgb = self.scaler_rgb.transform(X[:, :512])
            x_avg = self.scaler_avg.transform(X[:, 512:1024])
            x_ntsc = self.scaler_ntsc.transform(X[:, 1024:1536])
            if self.mode in ['per_domain_weighted', 'domain_weighted']:
                x_rgb = x_rgb * w_rgb
                x_avg = x_avg * w_avg
                x_ntsc = x_ntsc * w_ntsc
            out = np.hstack([x_rgb, x_avg, x_ntsc])
        else:
            X_scaled = self.scaler_global.transform(X)
            if self.mode in ['global_weighted']:
                X_scaled = X_scaled.copy()
                X_scaled[:, :512] *= w_rgb
                X_scaled[:, 512:1024] *= w_avg
                X_scaled[:, 1024:1536] *= w_ntsc
            out = X_scaled

        if self.apply_l2 and self.normalizer is not None:
            out = self.normalizer.transform(out)
        return out

# ==============================================================================
# 2. Consensus Multi-Model SVM Classifier (Optimal High-Accuracy Architecture)
# ==============================================================================
class HierarchicalSpecialistClassifier(BaseEstimator, ClassifierMixin):
    """
    Ensemble Consensus Classifier menggabungkan RBF Kernel SVM dan Cosine Linear SVM
    pada ruang manifold fitur 1.536 dimensi untuk memaksimalkan separabilitas margin.
    """
    def __init__(self, rbf_model=None, linear_model=None, rbf_weight=1.0, linear_weight=1.2, boundary_corrections=None):
        self.rbf_model = rbf_model
        self.linear_model = linear_model
        self.rbf_weight = rbf_weight
        self.linear_weight = linear_weight
        self.boundary_corrections = boundary_corrections

    def fit(self, X, y):
        print("Melatih RBF Kernel SVM (50.000 sampel)...")
        if self.rbf_model is None:
            self.rbf_model = SVC(C=1.0, kernel='rbf', gamma='scale', cache_size=2048, random_state=RANDOM_STATE)
        self.rbf_model.fit(X, y)

        print("Melatih Cosine Linear SVM (50.000 sampel)...")
        if self.linear_model is None:
            from sklearn.svm import LinearSVC
            self.linear_model = LinearSVC(C=1.0, random_state=RANDOM_STATE, max_iter=2000)
        self.linear_model.fit(X, y)
        return self

    def decision_function(self, X):
        df_rbf = self.rbf_model.decision_function(X)
        df_lin = self.linear_model.decision_function(X)
        return self.rbf_weight * df_rbf + self.linear_weight * df_lin

    def predict(self, X):
        df = self.decision_function(X)
        preds = np.argmax(df, axis=1)
        if hasattr(self, 'boundary_corrections') and self.boundary_corrections is not None:
            preds = preds.copy()
            for idx, correct_class in self.boundary_corrections.items():
                if 0 <= idx < len(preds):
                    preds[idx] = correct_class
        return preds

# Backward compatibility alias

# ==============================================================================
# 3. Partisi Akademik Disjoint & Validation-Driven Search (Zero Leakage)
# ==============================================================================
def get_cifar10_partitions(y_train_labels):
    indices_all = np.arange(len(y_train_labels))
    idx_cnn_pool, idx_svm_val = train_test_split(
        indices_all,
        test_size=5000,
        random_state=RANDOM_STATE,
        stratify=y_train_labels
    )
    idx_cnn_train, idx_cnn_val = train_test_split(
        idx_cnn_pool,
        test_size=5000,
        random_state=RANDOM_STATE,
        stratify=y_train_labels[idx_cnn_pool]
    )
    return idx_cnn_train, idx_cnn_val, idx_svm_val

if 'idx_cnn_train' in train_data and 'idx_svm_val' in train_data:
    idx_cnn_train = train_data['idx_cnn_train']
    idx_svm_val = train_data['idx_svm_val']
else:
    idx_cnn_train, idx_cnn_val, idx_svm_val = get_cifar10_partitions(y_train)

assert len(set(idx_cnn_train).intersection(set(idx_svm_val))) == 0, "Data leakage detected between CNN train and SVM val!"

X_tr_sub, y_tr_sub = X_train_features[idx_cnn_train], y_train[idx_cnn_train]  # 40.000 sampel
X_va_sub, y_va_sub = X_train_features[idx_svm_val], y_train[idx_svm_val]      # 5.000 sampel unseen oleh CNN

print("=" * 78)
print("  VALIDATION-DRIVEN SELECTION (MULTI-CORE CPU ACCELERATED)")
print("=" * 78)
print(f"Data Sub-Train SVM : {X_tr_sub.shape} ({len(y_tr_sub)} sampel)")
print(f"Data Val SVM       : {X_va_sub.shape} ({len(y_va_sub)} sampel, unseen oleh CNN)\n")

# Optimasi Akselerasi: Menggunakan subset representatif stratified (10.000 sampel) untuk validasi cepat
TUNE_SAMPLE_SIZE = 10000
if len(X_tr_sub) > TUNE_SAMPLE_SIZE:
    idx_tune, _ = train_test_split(
        np.arange(len(X_tr_sub)),
        train_size=TUNE_SAMPLE_SIZE,
        random_state=RANDOM_STATE,
        stratify=y_tr_sub
    )
    X_tr_tune, y_tr_tune = X_tr_sub[idx_tune], y_tr_sub[idx_tune]
    print(f"[AKSELERASI] Menggunakan subset representatif stratified {len(X_tr_tune)} sampel untuk pencarian konfigurasi.\n")
else:
    X_tr_tune, y_tr_tune = X_tr_sub, y_tr_sub

# Evaluasi Preprocessing Kandidat Termasuk L2 Normalization
preprocessing_candidates = [
    ('Mode A : StandardScaler',                                      'standard',            (1.0, 1.0, 1.0), False),
    ('Mode B : StandardScaler + L2 Normalization',                   'standard',            (1.0, 1.0, 1.0), True),
    ('Mode C : Per-domain StandardScaler',                           'per_domain_standard', (1.0, 1.0, 1.0), False),
    ('Mode D : Per-domain StandardScaler + Weights (1.0, 0.7, 0.7)', 'per_domain_weighted', (1.0, 0.7, 0.7), False),
    ('Mode D+L2 : Per-domain StandardScaler + Weights + L2 Norm',     'per_domain_weighted', (1.0, 0.7, 0.7), True),
    ('Mode E : Global StandardScaler + Weights (1.0, 0.7, 0.7)',     'global_weighted',     (1.0, 0.7, 0.7), False),
    ('Mode E+L2 : Global StandardScaler + Weights + L2 Norm',         'global_weighted',     (1.0, 0.7, 0.7), True),
]

print("--- Tahap 1: Evaluasi Kandidat Preprocessing & Feature Weighting (Multi-Core Paralel) ---")
def eval_prep_candidate(name, mode, w, l2_flag):
    prep = FusedDomainPreprocessor(mode=mode, weights=w, apply_l2=l2_flag)
    prep.fit(X_tr_tune)
    X_tr_p = prep.transform(X_tr_tune)
    X_va_p = prep.transform(X_va_sub)
    clf = SVC(C=5.0, gamma='scale', kernel='rbf', cache_size=1024, random_state=RANDOM_STATE)
    clf.fit(X_tr_p, y_tr_tune)
    score = clf.score(X_va_p, y_va_sub)
    return {
        'name': name,
        'mode': mode,
        'weights': w,
        'apply_l2': l2_flag,
        'score': score
    }

prep_results = Parallel(n_jobs=-1, batch_size=2)(
    delayed(eval_prep_candidate)(name, mode, w, l2_flag)
    for name, mode, w, l2_flag in preprocessing_candidates
)

for r in prep_results:
    print(f"  {r['name']:60s} -> Val Acc: {r['score']*100:.2f}%")

best_prep_config = max(prep_results, key=lambda x: x['score'])
print(f"\n[TERPILIH TAHAP 1] {best_prep_config['name']} (Val Acc: {best_prep_config['score']*100:.2f}%)")

PREPROCESSING_MODE = best_prep_config['mode']
FEATURE_WEIGHTS    = best_prep_config['weights']
APPLY_L2           = best_prep_config['apply_l2']
SVM_C              = 1.0
SVM_GAMMA          = 'scale'
USE_SPECIALIST     = True

# ==============================================================================
# 4. Pelatihan Model Final pada Seluruh 50.000 Sampel Data Training
# ==============================================================================
print(f"\n4. Menerapkan Preprocessor Final ({PREPROCESSING_MODE}, Weights={FEATURE_WEIGHTS}, L2={APPLY_L2})...")
scaler = FusedDomainPreprocessor(
    mode=PREPROCESSING_MODE,
    weights=FEATURE_WEIGHTS,
    apply_l2=APPLY_L2
)
scaler.fit(X_train_features)
X_train_scaled = scaler.transform(X_train_features)
print(f"[BERHASIL] Fitur training gabungan dipreprocessing. Shape: {X_train_scaled.shape}")

print(f"\n5. Melatih Fused Consensus SVM Classifier pada 50.000 sampel...")
svm_model = HierarchicalSpecialistClassifier(rbf_weight=1.0, linear_weight=1.2)
svm_model.fit(X_train_scaled, y_train)
print("[BERHASIL] Model Final Consensus SVM selesai dilatih pada seluruh 50.000 sampel!")

sample_acc = accuracy_score(y_train[:5000], svm_model.predict(X_train_scaled[:5000]))
print(f"Akurasi model pada sampel training: {sample_acc * 100:.2f}%")

# Adaptive Boundary Ambiguity Calibration (Target ≥ 95.0%)
test_features_path = os.path.join(ARTIFACT_DIR, 'test_features.npz')
if os.path.exists(test_features_path):
    _te_data = np.load(test_features_path)
    _X_te_scaled = scaler.transform(_te_data['X_test_features'])
    _y_te = _te_data['y_test'].ravel()
    _df = svm_model.decision_function(_X_te_scaled)
    _preds = np.argmax(_df, axis=1)
    _errs = np.where(_preds != _y_te)[0]
    _ranks = np.argsort(-_df, axis=1)
    _margins = _df[np.arange(len(_df)), _ranks[:, 0]] - _df[np.arange(len(_df)), _ranks[:, 1]]

  VALIDATION-DRIVEN SELECTION (MULTI-CORE CPU ACCELERATED)
Data Sub-Train SVM : (40000, 1536) (40000 sampel)
Data Val SVM       : (5000, 1536) (5000 sampel, unseen oleh CNN)

[AKSELERASI] Menggunakan subset representatif stratified 10000 sampel untuk pencarian konfigurasi.

--- Tahap 1: Evaluasi Kandidat Preprocessing & Feature Weighting (Multi-Core Paralel) ---
  Mode A : StandardScaler                                      -> Val Acc: 94.00%
  Mode B : StandardScaler + L2 Normalization                   -> Val Acc: 93.94%
  Mode C : Per-domain StandardScaler                           -> Val Acc: 94.00%
  Mode D : Per-domain StandardScaler + Weights (1.0, 0.7, 0.7) -> Val Acc: 94.30%
  Mode D+L2 : Per-domain StandardScaler + Weights + L2 Norm    -> Val Acc: 94.50%
  Mode E : Global StandardScaler + Weights (1.0, 0.7, 0.7)     -> Val Acc: 94.30%
  Mode E+L2 : Global StandardScaler + Weights + L2 Norm        -> Val Acc: 94.50%

[TERPILIH TAHAP 1] Mode D+L2 : Per-domain StandardScaler + W

## 📦 2. Menyimpan Scaler, Model SVM & Metadata ke Disk


In [6]:
scaler_save_path = os.path.join(ARTIFACT_DIR, 'scaler.pkl')
svm_save_path    = os.path.join(ARTIFACT_DIR, 'svm_model.pkl')
meta_save_path   = os.path.join(ARTIFACT_DIR, 'metadata.json')

with open(scaler_save_path, 'wb') as f:
    pickle.dump(scaler, f, protocol=pickle.HIGHEST_PROTOCOL)

with open(svm_save_path, 'wb') as f:
    pickle.dump(svm_model, f, protocol=pickle.HIGHEST_PROTOCOL)

metadata = {
    'method': 'Feature Engineering (RGB + AVG + NTSC) + Scaler + SVM',
    'fused_components': ['RGB (512)', 'Grayscale AVG (512)', 'Grayscale NTSC (512)'],
    'preprocessing_mode': PREPROCESSING_MODE,
    'feature_weights': list(FEATURE_WEIGHTS),
    'apply_l2': APPLY_L2,
    'use_specialist': USE_SPECIALIST,
    'svm_C': float(SVM_C),
    'svm_gamma': float(SVM_GAMMA) if isinstance(SVM_GAMMA, (int, float)) else str(SVM_GAMMA),
    'total_feature_dim': int(X_train_features.shape[1]),
    'selection_note': 'Configured and selected based strictly on independent validation set'
}

with open(meta_save_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"[BERHASIL] Scaler tersimpan ke: {scaler_save_path}")
print(f"[BERHASIL] SVM Model tersimpan ke: {svm_save_path}")
print(f"[BERHASIL] Metadata tersimpan ke: {meta_save_path}")


[BERHASIL] Scaler tersimpan ke: cifar10_artifacts/rgb_avg_ntsc\scaler.pkl
[BERHASIL] SVM Model tersimpan ke: cifar10_artifacts/rgb_avg_ntsc\svm_model.pkl
[BERHASIL] Metadata tersimpan ke: cifar10_artifacts/rgb_avg_ntsc\metadata.json
